# 05 — Emergency Response Time Regression
**RoadSentinel AI** — Trains Ridge and Random Forest regressors on EMS/911 dispatch records.
Predicts emergency vehicle arrival time (minutes) from severity level, borough, time of day, and call volume density.

In [ ]:
import os, sys
sys.path.insert(0, '..')
import pandas as pd, joblib
from src.regression import split_regression_data, train_ridge, train_rf_regressor, evaluate_and_plot_regression

## 1. Load EMS Incident Data

In [ ]:
ems_path = '../data/nyc_ems_response.csv'
if not os.path.exists(ems_path):
    from scripts.seed_data_generator import generate_all
    generate_all()

ems_df = pd.read_csv(ems_path)
print(f"EMS dataset loaded with {len(ems_df)} records.")
ems_df.head()

## 2. Leak-Safe Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = split_regression_data(ems_df)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## 3. Fit Ridge & Random Forest Regressors

In [ ]:
print("Training Ridge Regressor (with alpha GridSearch)...\n")
ridge_reg = train_ridge(X_train, y_train)

print("Training Random Forest Regressor...\n")
rf_reg = train_rf_regressor(X_train, y_train)

## 4. Evaluate & Plot Prediction Diagnostics (R², MAE, RMSE)

In [ ]:
print("Ridge Evaluation:")
evaluate_and_plot_regression(ridge_reg, X_test, y_test, save_path='../models/metrics/ridge_regression_metrics.png')

print("Random Forest Evaluation:")
metrics = evaluate_and_plot_regression(rf_reg, X_test, y_test, save_path='../models/metrics/regression_metrics.png')
joblib.dump(rf_reg, '../models/response_time_regressor.joblib')
print("Saved best regressor -> ../models/response_time_regressor.joblib")